In [1]:
from dataclasses import dataclass
from trainer import count_parameters, TrainConfig, train_waitk_student, TranslationDataset
import mlflow
import torch
from mamba_ssm import Mamba2
from evaluation import SimulMTEvaluator, MTQualityScorer, WaitKMamba2Adapter
from model_classes import WaitKMamba2MT, SimulMamba2Config

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("simulmt_waitk_mamba2_distillation")

<Experiment: artifact_location='/mlflow/artifacts/2', creation_time=1779451055489, experiment_id='2', last_update_time=1779451055489, lifecycle_stage='active', name='simulmt_waitk_mamba2_distillation', tags={}, trace_location=None, workspace='default'>

In [2]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

teacher_name = "./models/nllb_teacher"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)

In [3]:
mamba2_config = SimulMamba2Config(
    vocab_size=len(tokenizer),

    d_model=512,
    num_layers=24,

    d_state=128,
    d_conv=4,
    expand=2,
    headdim=64,
    ngroups=1,

    dropout=0.1,

    max_source_len=64,
    max_target_len=64,

    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,

    tie_embeddings=True,
    separate_source_target_embeddings=True,
)

mamba2 = WaitKMamba2MT(mamba2_config)
count_parameters(mamba2)

Total parameters:     303,718,016
Trainable parameters: 303,718,016
Frozen parameters:    0


{'total': 303718016, 'trainable': 303718016, 'frozen': 0}

In [7]:
dataset = TranslationDataset("./data/train_dataset.hdf5")

In [8]:
train_cfg = TrainConfig(
    epochs=5,
    short_epochs=False,
    batches_per_epoch=2000,
    batch_size=192,
    gradient_accumulation_steps=8,

    wait_k=10,

    use_kl_loss=True,
    use_dataset_ce_loss=True,

    kl_weight=1.0,
    dataset_ce_weight=1.0,

    lr=1e-4,
    use_amp=True,
)

train_waitk_student(
    student=mamba2,
    train_dataset=dataset,
    model_cfg=mamba2_config,
    train_cfg=train_cfg,
    device="cuda",
)

epoch 1/5:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 2/5:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 3/5:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 4/5:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 5/5:   0%|          | 0/14504 [00:00<?, ?it/s]

🏃 View run waitk-student at: http://localhost:5000/#/experiments/2/runs/16614573ca0241ef85bc0c9843abebf8
🧪 View experiment at: http://localhost:5000/#/experiments/2


In [4]:
def load_training_checkpoint(
    checkpoint_path: str,
    *,
    device: str = "cuda"
):
    """
    Load model, configs, optimizer, scaler and training progress
    from a full training checkpoint.

    Returns:
        student
        optimizer
        scaler
        model_cfg
        train_cfg
        state
        train_time
    """
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )

    model_cfg = SimulMamba2Config(**checkpoint["model_cfg"])
    train_cfg = TrainConfig(**checkpoint["train_cfg"])
    train_time = checkpoint["train_time"]

    student = torch.compile(WaitKMamba2MT(model_cfg))
    student.load_state_dict(checkpoint["model_state_dict"])
    student.to(device)
    optimizer = torch.optim.AdamW(
        student.parameters(),
        lr=train_cfg.lr,
        weight_decay=train_cfg.weight_decay,
        betas=(0.9, 0.98),
    )

    if "optimizer_state_dict" in checkpoint and checkpoint["optimizer_state_dict"] is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    scaler = torch.amp.GradScaler(
        enabled=train_cfg.use_amp and device == "cuda"
    )

    if "scaler_state_dict" in checkpoint and checkpoint["scaler_state_dict"] is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    state = {
        "epoch": checkpoint.get("epoch", 0),
        "global_step": checkpoint.get("global_step", checkpoint.get("step", 0)),
        "optimizer_step": checkpoint.get("optimizer_step", checkpoint.get("step", 0)),
    }

    return student, optimizer, scaler, model_cfg, train_cfg, state, train_time

In [5]:
mamba2, optimizer, scaler, model_cfg, train_cfg, state, train_time = load_training_checkpoint("./models/mamba_wait10.pt")

In [ ]:
eval_dataset = TranslationDataset(
    "./data/val_dataset.hdf5",
    lazy=False,
)

adapter = WaitKMamba2Adapter(
    model=mamba2,
    tokenizer=tokenizer,
    name="mamba2_wait_k",
    device="cuda",
    use_amp=True,
)

evaluator = SimulMTEvaluator(
    tokenizer=tokenizer,
    quality_scorer=MTQualityScorer(),
)

for k in [3, 5, 7, 9, 11, 13]:
    result = evaluator.evaluate(
        model=adapter,
        dataset=eval_dataset,
        wait_k=k,
        batch_size=1024,
        max_new_tokens=63
    )
    
    print(result.metrics)
    
    with open(f"./metrics/transformer_k{k}.json", "w") as file:
        json.dump(result.metrics, file)

Validating mamba2_wait_k, wait_k=3:   0%|          | 0/303 [00:00<?, ?it/s]